# K-Modes Clustering
## Template Notebook
### Partition-based clustering for categorical features

#### Load dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from kmodes.kmodes import KModes

df = pd.read_csv('PATH_TO_DATASET.csv')
df.head()

#### Select features

In [ ]:
X = df.drop(columns=['TARGET_COLUMN'])  # drop target - clustering is unsupervised

#### Encode / scale features

In [ ]:
X_encoded = X.copy()  # K-Modes operates directly on categorical columns

#### Find suitable number of clusters

In [ ]:
k_values = range(2, 9)
costs = []
for k in k_values:
    km = KModes(n_clusters=k, init='Huang', n_init=5, random_state=1265, verbose=0)
    km.fit(X_encoded)
    costs.append(km.cost_)

plt.plot(list(k_values), costs, marker='o')
plt.xlabel('k')
plt.ylabel('Cost')
plt.title('Elbow method')
plt.show()
# Elbow point: the k with the largest perpendicular distance from the line
# joining the first and last points of the cost curve
elbow_index = 0  # TODO: compute programmatically

In [ ]:
# Elbow point: the k with the largest perpendicular distance from the line
# joining the first and last point of the cost curve
k_arr = np.array(list(k_values), dtype=float)
c_arr = np.array(costs, dtype=float)

x = (k_arr - k_arr.min()) / (k_arr.max() - k_arr.min())
y = (c_arr - c_arr.min()) / (c_arr.max() - c_arr.min())

distances = np.abs((y[-1] - y[0]) * x - (x[-1] - x[0]) * y + x[-1] * y[0] - y[-1] * x[0])
distances /= np.sqrt((y[-1] - y[0]) ** 2 + (x[-1] - x[0]) ** 2)

best_k = list(k_values)[np.argmax(distances)]
print(f"Suitable number of clusters (elbow point): {best_k}")

#### Train final model

In [ ]:
best_k = list(k_values)[elbow_index]  # pick elbow point from the cost curve
model = KModes(n_clusters=best_k, init='Huang', n_init=5, random_state=1265, verbose=0)
cluster_labels = model.fit_predict(X_encoded)

#### Performance evaluation

In [ ]:
from sklearn.metrics import silhouette_score

score = silhouette_score(X_encoded, cluster_labels)
print('Score:', score)

#### Visualize results

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=1265)
X_2d = pca.fit_transform(X_encoded)
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=cluster_labels, cmap='tab10', alpha=0.7)
plt.title('Cluster Visualization (PCA)')
plt.show()